Cargar los datos (lectura y procesamiento de datos - Pandas)

In [110]:
'''

El presente script se encarga de analizar los datos de ventas, los inventarios y la satisfacción del cliente para una cadena de tiendas minoristas;
utilizando Pandas, para la manipulación de datos, y Numpy, para realizar cálculos estadísticos y simulaciones.
El objetivo es ayudar a optimizar el rendimiento de las tiendas a través del análisis de estos datos.

'''

import pandas as pd
import numpy as np

# Rutas absolutas solicitadas en el requerimiento 4
# ruta_ventas = '/workspace/sales.csv'
# ruta_inventarios = '/workspace/inventories.csv'
# ruta_satisfaccion = '/workspace/satisfaction.csv'
ruta_ventas = 'sales.csv'
ruta_inventarios = 'inventories.csv'
ruta_satisfaccion = 'satisfaction.csv'

# Cargar los DataFrames
df_sales = pd.read_csv(ruta_ventas)
df_inventarios = pd.read_csv(ruta_inventarios)
df_satisfaccion = pd.read_csv(ruta_satisfaccion)

# Eliminar filas con valores faltantes
df_sales.dropna(inplace=True)
df_inventarios.dropna(inplace=True)
df_satisfaccion.dropna(inplace=True)

print(df_sales.head(5))
print(df_inventarios.head(5))
print(df_satisfaccion.head(5))




   ID_Tienda    Producto  Cantidad_Vendida  Precio_Unitario Fecha_Venta
0          1  Producto A                20              100  2023-01-05
1          1  Producto B                15              200  2023-01-06
2          2  Producto A                30              100  2023-01-07
3          2  Producto C                25              300  2023-01-08
4          3  Producto A                10              100  2023-01-09
   ID_Tienda    Producto  Stock_Disponible Fecha_Actualización
0          1  Producto A                50          2023-01-05
1          1  Producto B                40          2023-01-06
2          2  Producto A                60          2023-01-07
3          2  Producto C                45          2023-01-08
4          3  Producto A                30          2023-01-09
   ID_Tienda  Satisfacción_Promedio Fecha_Evaluación
0          1                     85       2023-01-15
1          2                     90       2023-01-15
2          3                   

Exploración de datos (Pandas)

In [111]:
# Ventas totales por producto y por tienda.
tot_prod = df_sales.groupby(['Producto'])['Cantidad_Vendida'].sum()
print("Ventas totales por producto:")
print(tot_prod)

tot_tienda = df_sales.groupby(['ID_Tienda'])['Cantidad_Vendida'].sum()
print("Ventas totales por tienda:")
print(tot_tienda)

# Total_Ventas por tienda
df_sales['Total_Ventas'] = df_sales['Cantidad_Vendida'] * df_sales['Precio_Unitario']
ventas_tot_prod_tienda = df_sales.groupby(['ID_Tienda'])['Total_Ventas'].sum()
print("Total_Ventas por tienda:")
print(ventas_tot_prod_tienda)

# Resumen estadístico
print("Resumen estadístico de las ventas:")
print(df_sales.describe())

# Promedio de ventas por tienda y categoría de producto
promedio_ventas = df_sales.groupby(['ID_Tienda', 'Producto'])['Cantidad_Vendida'].mean()
print("Promedio de ventas por tienda y categoría de producto:")
print(promedio_ventas)


Ventas totales por producto:
Producto
Producto A    85
Producto B    75
Producto C    90
Name: Cantidad_Vendida, dtype: int64
Ventas totales por tienda:
ID_Tienda
1    35
2    55
3    50
4    60
5    50
Name: Cantidad_Vendida, dtype: int64
Total_Ventas por tienda:
ID_Tienda
1     5000
2    10500
3     9000
4    13000
5    13000
Name: Total_Ventas, dtype: int64
Resumen estadístico de las ventas:
       ID_Tienda  Cantidad_Vendida  Precio_Unitario  Total_Ventas
count  10.000000         10.000000        10.000000     10.000000
mean    3.000000         25.000000       190.000000   5050.000000
std     1.490712          9.128709        87.559504   3361.960407
min     1.000000         10.000000       100.000000   1000.000000
25%     2.000000         20.000000       100.000000   2625.000000
50%     3.000000         25.000000       200.000000   3500.000000
75%     4.000000         30.000000       275.000000   7875.000000
max     5.000000         40.000000       300.000000  10500.000000
Promedio

Análisis de inventarios (Pandas)

In [112]:
# Rotación de inventarios para cada tienda.
# merge para crear un df temporal entre ventas e inventarios para calcular la rotación
df_temp = pd.merge(df_inventarios, df_sales, on=['ID_Tienda', 'Producto'], how='left')
# print("DataFrame temporal para cálculo de rotación:")
# print(df_temp)

rotacion_calculada = df_temp['Cantidad_Vendida'] / df_temp['Stock_Disponible']
# print("Rotación de inventarios calculada:")
# print(rotacion_calculada)

df_inventarios['Rotacion_Inventario'] = rotacion_calculada
print("Rotación de inventarios por tienda:")
print(df_inventarios)

criticos = df_inventarios[df_inventarios['Rotacion_Inventario']<0.1]
print("Productos con rotación crítica (menos de 0.1):")
print(criticos)


Rotación de inventarios por tienda:
   ID_Tienda    Producto  Stock_Disponible Fecha_Actualización  \
0          1  Producto A                50          2023-01-05   
1          1  Producto B                40          2023-01-06   
2          2  Producto A                60          2023-01-07   
3          2  Producto C                45          2023-01-08   
4          3  Producto A                30          2023-01-09   
5          3  Producto B                80          2023-01-10   
6          4  Producto C                70          2023-01-11   
7          4  Producto A                50          2023-01-12   
8          5  Producto B                40          2023-01-13   
9          5  Producto C                60          2023-01-14   

   Rotacion_Inventario  
0             0.400000  
1             0.375000  
2             0.500000  
3             0.555556  
4             0.333333  
5             0.500000  
6             0.500000  
7             0.500000  
8           

Satisfacción del cliente (Pandas)

In [113]:
df_temp = pd.merge(df_temp, df_satisfaccion, on='ID_Tienda', how='left')
print("DataFrame temporal para análisis de satisfacción:")
print(df_temp)

baja_satisfaccion = df_temp[df_temp['Satisfacción_Promedio']<60]
print("Tiendas con baja satisfacción (menos de 60%):")
print(baja_satisfaccion)

DataFrame temporal para análisis de satisfacción:
   ID_Tienda    Producto  Stock_Disponible Fecha_Actualización  \
0          1  Producto A                50          2023-01-05   
1          1  Producto B                40          2023-01-06   
2          2  Producto A                60          2023-01-07   
3          2  Producto C                45          2023-01-08   
4          3  Producto A                30          2023-01-09   
5          3  Producto B                80          2023-01-10   
6          4  Producto C                70          2023-01-11   
7          4  Producto A                50          2023-01-12   
8          5  Producto B                40          2023-01-13   
9          5  Producto C                60          2023-01-14   

   Cantidad_Vendida  Precio_Unitario Fecha_Venta  Total_Ventas  \
0                20              100  2023-01-05          2000   
1                15              200  2023-01-06          3000   
2                30      

Se observa que para la tienda 5 se tiene mayor cantidad de Stock para los productos B y C por lo que se recomienda disminuir su Stock para evitar que el producto pase de moda o se caduque.

Operaciones con Numpy

In [ ]:
arr_tot_ventas = df_sales['Total_Ventas'].to_numpy()
media_ventas = np.mean(arr_tot_ventas)
print(f"Media de las ventas totales por tienda: {media_ventas:.2f}")
std_dev_ventas = np.std(arr_tot_ventas)
print(f"Desviación estándar de las ventas totales por tienda: {std_dev_ventas:.2f}")

# Generando datos de ventas simulados para los próximos 10 días utilizando una distribución normal
# Establecemos la semilla para reproducibilidad
np.random.seed(42)

ventas_simuladas = np.random.normal(loc=media_ventas, scale=std_dev_ventas, size=df_sales.shape[0])
print("Ventas simuladas para los próximos 10 días:")
print(ventas_simuladas)


Media de las ventas totales por tienda: 5050.00
Desviación estándar de las ventas totales por tienda: 3189.44
Número de filas en df_sales: 10
Ventas simuladas para el próximo mes:
[ 6634.23784573  4609.01490364  7115.76093733  9907.60577603
  4303.18287048  4303.23523392 10086.79771077  7497.68371242
  3552.64163948  6780.46036522]
